In [2]:
import pandas as pd
from openpyxl import load_workbook

input_file = "C:/Users/Ex0164/Tushar vats/Copy of 1401 BOM.xlsx"
output_file = "series_part_mapping_1.xlsx"

try:
    wb = load_workbook(input_file, data_only=True)
    ws = wb.active
    
    rows = []
    current_series = None

    # We iterate through rows while keeping track of the cell objects
    for row in ws.iter_rows(min_row=2):
        # Extract values for logic
        values = [cell.value for cell in row if cell.value not in (None, "Grand Total")]
        
        # We need the first cell object specifically to check for Bold formatting
        first_cell = row[0]
        is_bold = first_cell.font.bold if first_cell.font else False
        
        # Extract numeric values
        nums = [v for v in values if isinstance(v, (int, float))]
        texts = [str(v).strip() for v in values if isinstance(v, str)]

        if not texts or not nums:
            continue

        label = texts[0]
        qty = int(nums[-1])

        # NEW LOGIC: If it's bold OR qty > 1, it's a parent
        if is_bold or qty > 1:
            current_series = label
        
        # If it's NOT bold AND qty == 1, it's a child part
        elif qty == 1 and current_series:
            rows.append({
                "Series": current_series,
                "Part": label, 
                "Count": 1
            })

    df = pd.DataFrame(rows)
    
    if not df.empty:
        df.to_excel(output_file, index=False)
        print(f"Success! Processed {len(df)} parts into their respective series.")
    else:
        print("Processing complete, but no child parts were mapped.")

except Exception as e:
    print(f"An error occurred: {e}")

new wxcel file created 114351 rows


In [ ]:
import pandas as pd
from openpyxl import load_workbook

input_file = "C:/Users/Ex0164/Tushar vats/Copy of 1401 BOM.xlsx"
output_file = "series_part_mapping_final.xlsx"

try:
    print("Loading workbook... (Large file, please wait)")
    wb = load_workbook(input_file, data_only=True)
    ws = wb.active
    
    rows = []
    current_parent = None
    child_remaining = 0

    print("Processing rows...")
    # Iterating through rows
    for row in ws.iter_rows(min_row=2):
        # Identify key cells (Adjust index if Label is not Col A and Qty is not Col B)
        label_cell = row[0]
        qty_cell = row[1]
        
        label = str(label_cell.value).strip() if label_cell.value is not None else ""
        if not label or "Grand Total" in label:
            continue

        # Check for Bold (Parent indicator)
        is_bold = label_cell.font.bold if label_cell.font else False
        
        try:
            qty = int(qty_cell.value) if qty_cell.value is not None else 0
        except (ValueError, TypeError):
            qty = 0

        # LOGIC ENGINE:
        if is_bold:
            # This is a Parent. Update the current parent and set the countdown.
            current_parent = label
            child_remaining = qty
            # We don't add the parent to the 'Part' list, just store it as the header.
            continue 
        
        elif child_remaining > 0:
            # This is a Child. Map it to the active parent and decrement the count.
            rows.append({
                "Parent Series": current_parent,
                "Child Part": label,
                "Status": "Mapped"
            })
            child_remaining -= 1
            
        else:
            # This handles rows that are neither bold parents nor within a parent's count
            # Helpful for debugging data gaps
            if label:
                rows.append({
                    "Parent Series": "UNASSOCIATED",
                    "Child Part": label,
                    "Status": "Review Needed"
                })

    # Exporting
    df = pd.DataFrame(rows)
    df.to_excel(output_file, index=False)
    
    print(f"Done! Processed {len(df)} associations.")
    if child_remaining > 0:
        print(f"Warning: The last parent expected {child_remaining} more children than were found.")

except Exception as e:
    print(f"An error occurred: {e}")

In [ ]:
import pandas as pd
from openpyxl import load_workbook

input_file = "C:/Users/Ex0164/Tushar vats/Copy of 1401 BOM.xlsx"
output_file = "series_part_mapping_final.xlsx"

def has_bold_cell(row):
    """Checks if any cell in the row has bold formatting."""
    for cell in row:
        if cell.font and cell.font.bold:
            return True
    return False

try:
    print("Loading workbook...")
    wb = load_workbook(input_file, data_only=True)
    ws = wb.active
    
    rows = []
    current_parent = "Unknown/Start"
    child_remaining = 0

    print("Processing rows...")
    # Using min_row=2 to skip headers
    for i, row in enumerate(ws.iter_rows(min_row=2), start=2):
        # 1. Extract values
        label = str(row[0].value).strip() if row[0].value is not None else ""
        
        # Get Qty (assuming it's in Column B / index 1)
        try:
            qty_val = row[1].value
            qty = int(float(qty_val)) if qty_val is not None else 0
        except (ValueError, TypeError):
            qty = 0

        if not label or "TOTAL" in label.upper():
            continue

        # 2. Stronger Parent Detection
        # A row is a parent if it's BOLD or if it has a quantity but we aren't currently filling a parent's quota
        is_bold = has_bold_cell(row)
        
        if is_bold and qty > 0:
            current_parent = label
            child_remaining = qty
            continue # Move to next row to start collecting children
        
        # 3. Association Logic
        if child_remaining > 0:
            rows.append({
                "Parent Series": current_parent,
                "Child Part": label,
                "Child_Qty_Check": 1
            })
            child_remaining -= 1
        else:
            # This logic captures parents that might not be bold but have children
            if qty > 0:
                current_parent = label
                child_remaining = qty
            else:
                rows.append({
                    "Parent Series": "ORPHAN",
                    "Child Part": label,
                    "Child_Qty_Check": 0
                })

    df = pd.DataFrame(rows)
    df.to_excel(output_file, index=False)
    print(f"Done! Created {output_file} with {len(df)} rows.")

except Exception as e:
    print(f"Error occurred: {e}")

In [ ]:
import pandas as pd
from openpyxl import load_workbook

input_file = "C:/Users/Ex0164/Tushar vats/Copy of 1401 BOM.xlsx"
output_file = "series_part_mapping_final.xlsx"

try:
    wb = load_workbook(input_file, data_only=True)
    ws = wb.active

    rows = []

    current_parent = None
    expected_total_qty = 0
    running_child_qty = 0

    for i, row in enumerate(ws.iter_rows(min_row=2), start=2):

        label = str(row[0].value).strip() if row[0].value else ""

        try:
            qty_val = row[-1].value   # ✅ LAST COLUMN
            qty = int(float(qty_val)) if qty_val is not None else 0
        except (ValueError, TypeError):
            qty = 0

        if not label or "TOTAL" in label.upper():
            continue

        is_bold = bool(row[0].font and row[0].font.bold)  # ✅ ONLY LABEL CELL

        # 🔹 SERIES ROW
        if is_bold:
            if current_parent and running_child_qty != expected_total_qty:
                print(
                    f"⚠️ Qty mismatch for series '{current_parent}' "
                    f"(expected {expected_total_qty}, got {running_child_qty}) "
                    f"before row {i}"
                )

            current_parent = label
            expected_total_qty = qty
            running_child_qty = 0
            continue

        # 🔹 PART ROW
        if current_parent:
            rows.append({
                "Parent Series": current_parent,
                "Child Part": label,
                "Child Qty": qty
            })
            running_child_qty += qty
        else:
            print(f"⚠️ Orphan part at row {i}: {label}")

    # Final validation
    if current_parent and running_child_qty != expected_total_qty:
        print(
            f"⚠️ Qty mismatch for series '{current_parent}' "
            f"(expected {expected_total_qty}, got {running_child_qty}) at EOF"
        )

    df = pd.DataFrame(rows)
    df.to_excel(output_file, index=False)

    print(f"Done! Created {output_file} with {len(df)} rows.")

except Exception as e:
    print(f"Error occurred: {e}")


In [ ]:
print(
    f"Row {i} | "
    f"A={row[0].value} | "
    f"A.bold={row[0].font.bold} | "
    f"LastCol={row[-1].value}"
)


In [ ]:
import pandas as pd
from openpyxl import load_workbook

# ==========================
# FILE PATHS
# ==========================
input_file = "C:/Users/Ex0164/Tushar vats/Copy of 1401 BOM.xlsx"
output_file = "series_part_mapping_final.xlsx"

try:
    print("Loading workbook...")
    wb = load_workbook(input_file, data_only=True)
    ws = wb.active

    rows = []

    current_series = None
    expected_series_qty = 0
    running_part_qty = 0

    print("Processing BOM...")

    # Skip header row
    for i, row in enumerate(ws.iter_rows(min_row=2), start=2):

        # --- Read label (Series / Part)
        label = str(row[0].value).strip() if row[0].value else ""

        # --- Read quantity from LAST column (as per your Excel)
        try:
            qty_val = row[-1].value
            qty = int(float(qty_val)) if qty_val is not None else 0
        except (ValueError, TypeError):
            qty = 0

        # --- Skip junk rows
        if not label or "TOTAL" in label.upper():
            continue

        # ==========================
        # SERIES ROW LOGIC
        # ==========================
        # Series rows have qty > 1 (e.g., 7 in your image)
        if qty > 1:
            # Validate previous series
            if current_series and running_part_qty != expected_series_qty:
                print(
                    f"⚠️ Qty mismatch for series '{current_series}' "
                    f"(expected {expected_series_qty}, got {running_part_qty}) "
                    f"before row {i}"
                )

            current_series = label
            expected_series_qty = qty
            running_part_qty = 0
            continue

        # ==========================
        # PART ROW LOGIC
        # ==========================
        if current_series:
            rows.append({
                "Parent Series": current_series,
                "Child Part": label,
                "Child Qty": qty
            })
            running_part_qty += qty
        else:
            print(f"⚠️ Orphan part at row {i}: {label}")

    # ==========================
    # FINAL VALIDATION
    # ==========================
    if current_series and running_part_qty != expected_series_qty:
        print(
            f"⚠️ Qty mismatch for series '{current_series}' "
            f"(expected {expected_series_qty}, got {running_part_qty}) at EOF"
        )

    # ==========================
    # EXPORT
    # ==========================
    df = pd.DataFrame(rows)
    df.to_excel(output_file, index=False)

    print(f"✅ Done! Created {output_file} with {len(df)} rows.")

except Exception as e:
    print(f"❌ Error occurred: {e}")


In [ ]:
import pandas as pd
from openpyxl import load_workbook

input_file = "C:/Users/Ex0164/Tushar vats/Copy of 1401 BOM.xlsx"
output_file = "series_part_mapping_refined.xlsx"

try:
    print("Loading workbook... (114k rows may take a moment)")
    # Using data_only=True to read formula results
    wb = load_workbook(input_file, data_only=True)
    ws = wb.active

    rows = []
    current_series = None
    expected_series_qty = 0
    running_part_qty = 0

    print("Processing BOM structure...")

    for i, row in enumerate(ws.iter_rows(min_row=1), start=1):
        # Ensure row has at least 2 columns to avoid index errors
        if len(row) < 2:
            continue

        label_cell = row[0]
        qty_cell = row[1] # Using Column B based on your 2-column update

        label = str(label_cell.value).strip() if label_cell.value else ""
        
        try:
            # Handle potential decimals and convert to integer
            qty_val = qty_cell.value
            qty = int(float(qty_val)) if qty_val is not None else 0
        except (ValueError, TypeError):
            qty = 0

        if not label or "TOTAL" in label.upper():
            continue

        # Check for BOLD formatting (The primary Parent indicator)
        is_bold = label_cell.font.bold if label_cell.font else False

        # ==========================
        # PARENT LOGIC
        # ==========================
        if is_bold:
            # Check if the previous parent's math was correct before switching
            if current_series and running_part_qty != expected_series_qty:
                print(f"⚠️ Mismatch: '{current_series}' at row {i-1}. "
                      f"Expected {expected_series_qty}, but sum was {running_part_qty}")

            current_series = label
            expected_series_qty = qty
            running_part_qty = 0
            continue 

        # ==========================
        # CHILD LOGIC
        # ==========================
        elif current_series:
            rows.append({
                "Parent Series": current_series,
                "Child Part": label,
                "Child Qty": qty
            })
            running_part_qty += qty
            
            # Auto-reset if the sum is exactly met
            if running_part_qty >= expected_series_qty:
                current_series = None
                expected_series_qty = 0
                running_part_qty = 0
        else:
            # Captures rows that aren't bold and don't belong to a sum
            print(f"⚠️ Orphan part at row {i}: {label}")

    # Final Export
    df = pd.DataFrame(rows)
    df.to_excel(output_file, index=False)
    print(f"✅ Success! Created {output_file} with {len(df)} mapped parts.")

except Exception as e:
    print(f"❌ Error occurred: {e}")

In [ ]:
import pandas as pd
from openpyxl import load_workbook

input_file = "C:/Users/Ex0164/Tushar vats/Copy of 1401 BOM.xlsx"
output_file = "series_part_mapping_final.xlsx"

def is_bold_row(row):
    # Series text is in column A
    return row[0].font and row[0].font.bold

try:
    print("Loading workbook...")
    wb = load_workbook(input_file, data_only=True)
    ws = wb.active

    rows = []

    current_series = None
    expected_series_qty = 0
    running_part_qty = 0

    print("Processing BOM...")

    for i, row in enumerate(ws.iter_rows(min_row=2), start=2):

        label = str(row[0].value).strip() if row[0].value else ""

        try:
            qty = int(float(row[-1].value)) if row[-1].value is not None else 0
        except (ValueError, TypeError):
            qty = 0

        if not label or "TOTAL" in label.upper():
            continue

        # ==========================
        # SERIES ROW (BOLD)
        # ==========================
        if is_bold_row(row):
            # Validate previous series before starting new one
            if current_series and running_part_qty != expected_series_qty:
                print(
                    f"⚠️ Qty mismatch for series '{current_series}' "
                    f"(expected {expected_series_qty}, got {running_part_qty}) "
                    f"before row {i}"
                )

            current_series = label
            expected_series_qty = qty
            running_part_qty = 0
            continue

        # ==========================
        # PART ROW (NON-BOLD)
        # ==========================
        if not current_series:
            print(f"⚠️ Orphan part at row {i}: {label}")
            continue

        rows.append({
            "Parent Series": current_series,
            "Child Part": label,
            "Child Qty": qty
        })

        running_part_qty += qty

        # Optional early close if quantities already match
        if running_part_qty == expected_series_qty:
            # Series block completed naturally
            continue

        if running_part_qty > expected_series_qty:
            print(
                f"❌ Qty overflow for series '{current_series}' "
                f"(expected {expected_series_qty}, exceeded at row {i})"
            )

    # ==========================
    # FINAL SERIES VALIDATION
    # ==========================
    if current_series and running_part_qty != expected_series_qty:
        print(
            f"⚠️ Qty mismatch for series '{current_series}' "
            f"(expected {expected_series_qty}, got {running_part_qty}) at EOF"
        )

    df = pd.DataFrame(rows)
    df.to_excel(output_file, index=False)

    print(f"✅ Done! Created {output_file} with {len(df)} rows.")

except Exception as e:
    print(f"❌ Error occurred: {e}")


In [ ]:
import pandas as pd
from openpyxl import load_workbook

input_file = "C:/Users/Ex0164/Tushar vats/Copy of 1401 BOM.xlsx"
output_file = "series_part_mapping_synchronized.xlsx"

try:
    print("Loading workbook... (114k rows)")
    wb = load_workbook(input_file, data_only=True)
    ws = wb.active

    rows = []
    current_series = None
    remaining_balance = 0 # This is the "7" that counts down to 0

    print("Processing with Countdown Logic...")

    for i, row in enumerate(ws.iter_rows(min_row=1), start=1):
        if len(row) < 2:
            continue

        label_cell = row[0]
        qty_cell = row[1]

        label = str(label_cell.value).strip() if label_cell.value else ""
        
        try:
            qty = int(float(qty_cell.value)) if qty_cell.value is not None else 0
        except (ValueError, TypeError):
            qty = 0

        if not label or "TOTAL" in label.upper():
            continue

        is_bold = label_cell.font.bold if label_cell.font else False

        # --- STEP 1: Identify New Series (The Bold '7') ---
        if is_bold:
            # If the previous series didn't finish correctly, log it
            if remaining_balance > 0:
                print(f"⚠️ Warning: Series '{current_series}' finished early at row {i-1}. Missing {remaining_balance} count.")

            current_series = label
            remaining_balance = qty
            continue 

        # --- STEP 2: Process Non-Bold Parts ---
        elif current_series:
            rows.append({
                "Parent Series": current_series,
                "Child Part": label,
                "Child Qty": qty
            })
            
            # --- STEP 3: The Subtraction Logic ---
            remaining_balance -= qty
            
            # Once balance hits 0, we have finished this series
            if remaining_balance <= 0:
                current_series = None
                remaining_balance = 0
        
        else:
            print(f"⚠️ Row {i} has no parent series: {label}")

    # Export
    df = pd.DataFrame(rows)
    df.to_excel(output_file, index=False)
    print(f"✅ Success! Created {output_file} with {len(df)} mapped parts.")

except Exception as e:
    print(f"❌ Error: {e}")

In [ ]:
import pandas as pd
from openpyxl import load_workbook

# =========================
# FILE PATHS
# =========================
input_file = "C:/Users/Ex0164/Tushar vats/Copy of 1401 BOM.xlsx"
output_file = "series_part_mapping_final.xlsx"

def is_bold_row(row):
    """Series row is identified ONLY by bold text in column A"""
    cell = row[0]
    return cell.font is not None and cell.font.bold is True

try:
    print("Loading workbook...")
    wb = load_workbook(input_file, data_only=True)
    ws = wb.active

    rows = []

    current_series = None
    remaining_qty = 0

    print("Processing BOM...")

    for i, row in enumerate(ws.iter_rows(min_row=2), start=2):

        # -------- Read label --------
        label = str(row[0].value).strip() if row[0].value else ""

        # -------- Read qty (LAST column) --------
        try:
            qty_val = row[-1].value
            qty = int(float(qty_val)) if qty_val is not None else 0
        except (ValueError, TypeError):
            qty = 0

        # -------- Skip junk --------
        if not label or "TOTAL" in label.upper():
            continue

        # =========================
        # SERIES ROW (BOLD)
        # =========================
        if is_bold_row(row):

            # If previous series not finished → ERROR
            if current_series and remaining_qty != 0:
                raise ValueError(
                    f"Series '{current_series}' incomplete before row {i}. "
                    f"Remaining qty = {remaining_qty}"
                )

            current_series = label
            remaining_qty = qty

            if remaining_qty <= 0:
                raise ValueError(
                    f"Invalid series quantity at row {i}: {remaining_qty}"
                )

            continue

        # =========================
        # PART ROW (NON-BOLD)
        # =========================
        if not current_series:
            raise ValueError(f"Orphan part '{label}' at row {i}")

        rows.append({
            "Parent Series": current_series,
            "Child Part": label,
            "Child Qty": qty
        })

        remaining_qty -= qty

        if remaining_qty < 0:
            raise ValueError(
                f"Quantity overflow in series '{current_series}' at row {i}"
            )

        # If remaining_qty == 0 → series automatically closes
        # Next bold row will start a new one

    # =========================
    # FINAL VALIDATION
    # =========================
    if current_series and remaining_qty != 0:
        raise ValueError(
            f"Series '{current_series}' incomplete at EOF. "
            f"Remaining qty = {remaining_qty}"
        )

    # =========================
    # EXPORT
    # =========================
    df = pd.DataFrame(rows)
    df.to_excel(output_file, index=False)

    print(f"✅ Done! Created {output_file} with {len(df)} rows.")

except Exception as e:
    print(f"❌ ERROR: {e}")


In [ ]:
import pandas as pd
from openpyxl import load_workbook

input_file = "C:/Users/Ex0164/Tushar vats/Copy of 1401 BOM.xlsx"
output_file = "series_part_mapping_synchronized.xlsx"

try:
    print("Loading workbook... (114k rows)")
    wb = load_workbook(input_file, data_only=True)
    ws = wb.active

    rows = []
    current_series = None
    remaining_balance = 0 

    print("Processing with Countdown Logic starting from Row 2...")

    # min_row=2 skips the header row entirely
    for i, row in enumerate(ws.iter_rows(min_row=2), start=2):
        if len(row) < 2:
            continue

        label_cell = row[0]
        qty_cell = row[1]

        label = str(label_cell.value).strip() if label_cell.value else ""
        
        try:
            qty = int(float(qty_cell.value)) if qty_cell.value is not None else 0
        except (ValueError, TypeError):
            qty = 0

        # Skip rows that are empty or just "Total" labels
        if not label or "TOTAL" in label.upper():
            continue

        is_bold = label_cell.font.bold if label_cell.font else False

        # --- STEP 1: Identify New Series (The Bold Parent) ---
        if is_bold:
            # Alert if the previous series didn't add up correctly
            if current_series and remaining_balance != 0:
                print(f"⚠️ Warning: Series '{current_series}' mismatch at row {i-1}. Balance: {remaining_balance}")

            current_series = label
            remaining_balance = qty
            continue 

        # --- STEP 2: Process Non-Bold Child Parts ---
        elif current_series and remaining_balance > 0:
            rows.append({
                "Parent Series": current_series,
                "Child Part": label,
                "Child Qty": qty
            })
            
            # Subtraction Logic: 7 - 1 = 6... 6 - 4 = 2... 2 - 2 = 0
            remaining_balance -= qty
            
            # Reset once the series is mathematically complete
            if remaining_balance <= 0:
                current_series = None
                remaining_balance = 0
        
        else:
            # This captures rows that appear outside of a bold parent's "budget"
            print(f"⚠️ Row {i} is unassociated: {label}")

    # Export
    df = pd.DataFrame(rows)
    df.to_excel(output_file, index=False)
    print(f"✅ Success! Created {output_file} with {len(df)} mapped parts.")

except Exception as e:
    print(f"❌ Error: {e}")

In [ ]:
import pandas as pd
from openpyxl import load_workbook

input_file = "C:/Users/Ex0164/Tushar vats/Copy of 1401 BOM.xlsx"
output_file = "series_part_mapping_final.xlsx"

try:
    print("Loading workbook... (Checking 114k rows)")
    wb = load_workbook(input_file, data_only=True)
    ws = wb.active

    rows = []
    current_parent = None
    remaining_balance = 0 

    # Starting from Row 2 to skip "Row Labels" and "Count of No.Off"
    for i, row in enumerate(ws.iter_rows(min_row=2), start=2):
        if len(row) < 2:
            continue

        label_cell = row[0]
        qty_cell = row[1] # Based on your image, Qty is in Column B

        # Clean the label (remove leading spaces and hierarchy symbols like [-])
        raw_label = str(label_cell.value).strip() if label_cell.value else ""
        label = raw_label.replace("", "").replace("- ", "").replace("+ ", "").strip()
        
        try:
            qty = int(float(qty_cell.value)) if qty_cell.value is not None else 0
        except:
            qty = 0

        if not label or "TOTAL" in label.upper():
            continue

        # PARENT DETECTION: 
        # 1. Is it Bold? 
        # 2. Does the raw text contain the pivot hierarchy symbol?
        is_bold = label_cell.font.bold if label_cell.font else False
        is_pivot_parent = "" in raw_label or raw_label.startswith("-")

        if (is_bold or is_pivot_parent) and qty > 0:
            # If the last parent didn't finish correctly, we log it but move on
            if current_parent and remaining_balance != 0:
                print(f"⚠️ Row {i-1}: Group '{current_parent}' closed with {remaining_balance} remaining.")

            current_parent = label
            remaining_balance = qty
            continue 

        # CHILD ASSOCIATION:
        elif current_parent and remaining_balance > 0:
            rows.append({
                "Parent Series": current_parent,
                "Child Part": label,
                "Child Qty": qty
            })
            
            # The "7 - 1 - 4 - 2 = 0" Logic
            remaining_balance -= qty
            
            # Reset once the parent's quota is satisfied
            if remaining_balance <= 0:
                current_parent = None
                remaining_balance = 0
        
        else:
            # This catches rows that fall between groups
            if label:
                print(f"⚠️ Row {i} is unassociated: {label}")

    # Export to Excel
    df = pd.DataFrame(rows)
    df.to_excel(output_file, index=False)
    print(f"✅ Success! Mapped {len(df)} parts correctly.")

except Exception as e:
    print(f"❌ Critical Error: {e}")

In [ ]:
import pandas as pd
from openpyxl import load_workbook

# =========================
# FILE PATHS
# =========================
INPUT_FILE = "C:/Users/Ex0164/Tushar vats/Copy of 1401 BOM.xlsx"
OUTPUT_FILE = "series_part_mapping_final.xlsx"

def is_bold(cell):
    """Safely check if a cell is bold"""
    return cell.font is not None and cell.font.bold is True

try:
    wb = load_workbook(INPUT_FILE, data_only=True)
    ws = wb.active

    output_rows = []

    current_parent = None
    expected_qty = 0
    running_qty = 0

    for row_idx, row in enumerate(ws.iter_rows(min_row=2), start=2):

        # ---- Read label (Column A) ----
        label_cell = row[0]
        label = str(label_cell.value).strip() if label_cell.value else ""

        # ---- Read quantity (LAST column) ----
        try:
            qty_cell = row[-1].value
            qty = int(float(qty_cell)) if qty_cell is not None else 0
        except (ValueError, TypeError):
            qty = 0

        # ---- Skip junk ----
        if not label or "TOTAL" in label.upper():
            continue

        # =========================
        # PARENT ROW (BOLD)
        # =========================
        if is_bold(label_cell):

            # Parent encountered before previous parent closed
            if current_parent and running_qty != expected_qty:
                raise ValueError(
                    f"❌ Parent '{current_parent}' incomplete before row {row_idx}. "
                    f"Expected {expected_qty}, got {running_qty}"
                )

            current_parent = label
            expected_qty = qty
            running_qty = 0

            if expected_qty <= 0:
                raise ValueError(
                    f"❌ Invalid quantity for parent '{label}' at row {row_idx}"
                )

            continue

        # =========================
        # CHILD ROW (NON-BOLD)
        # =========================
        if not current_parent:
            raise ValueError(
                f"❌ Orphan child '{label}' at row {row_idx}"
            )

        output_rows.append({
            "Parent": current_parent,
            "Child": label,
            "Child_Qty": qty
        })

        running_qty += qty

        if running_qty > expected_qty:
            raise ValueError(
                f"❌ Quantity overflow in parent '{current_parent}' at row {row_idx}. "
                f"Expected {expected_qty}, exceeded with {running_qty}"
            )

        # If running_qty == expected_qty → parent is logically closed
        # Next bold row will start a new parent

    # =========================
    # FINAL VALIDATION
    # =========================
    if current_parent and running_qty != expected_qty:
        raise ValueError(
            f"❌ Parent '{current_parent}' incomplete at EOF. "
            f"Expected {expected_qty}, got {running_qty}"
        )

    # =========================
    # EXPORT
    # =========================
    df = pd.DataFrame(output_rows)
    df.to_excel(OUTPUT_FILE, index=False)

    print(f"✅ SUCCESS: {len(df)} rows written to {OUTPUT_FILE}")

except Exception as e:
    print(str(e))
